## Load prediction data


In [44]:

import numpy as np
import matplotlib.pyplot as plt
import os
from scipy.stats import gaussian_kde  # Used for kernel density estimation
# Configure Chinese font display
plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

def load_saved_results(save_root):
    """Load all saved network prediction results"""
    all_results = {}
    if not os.path.exists(save_root):
        raise FileNotFoundError(f"Result directory does not exist: {save_root}")
    
    for filename in os.listdir(save_root):
        if filename.endswith("_predictions.npz"):
            net_type = filename.split("_predictions.npz")[0]
            file_path = os.path.join(save_root, filename)
            data = np.load(file_path, allow_pickle=True)
            all_results[net_type] = {
                "x": data["x"],
                "y_true": data["y_true"],
                "y_pred": data["y_pred"],
                "freq": data["freq"]
            }
    return all_results

save_root = '../results/predictions'  # Root directory for saving results
# Load results and plot
all_results = load_saved_results(save_root)


## 2D result comparison for two networks


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

# ========== Plot style configuration ==========
plt.rcParams['font.family'] = 'Times New Roman'
plt.rcParams['xtick.major.size'] = 2.
plt.rcParams['ytick.major.size'] = 2.5
plt.rcParams['xtick.minor.size'] = 1.5
plt.rcParams['ytick.minor.size'] = 1.5
plt.rcParams['xtick.major.width'] = 0.5
plt.rcParams['ytick.major.width'] = 0.5
plt.rcParams['xtick.minor.width'] = 0.5
plt.rcParams['ytick.minor.width'] = 0.5
plt.rcParams['lines.linewidth'] = 1.5
plt.rcParams['lines.markersize'] = 3.5
plt.rcParams['font.size'] = 12
plt.rcParams['xtick.labelsize'] = 22
plt.rcParams['ytick.labelsize'] = 22
plt.rcParams['axes.labelsize'] = 22  # Font size for x, y labels
plt.rcParams['axes.titlesize'] = 22
plt.rcParams['legend.fontsize'] = 22
plt.rcParams['legend.title_fontsize'] = 24
plt.rcParams["savefig.bbox"] = 'tight'
plt.rcParams["savefig.pad_inches"] = 0.1
plt.rcParams['image.cmap'] = 'jet_r'
plt.rcParams['figure.dpi'] = 600


def plot_with_error_comparison(all_results, sample_idx=0, save_dir="FiguresPlot"):
    """
    Plot 4-row 3-column spatial distribution comparison chart (implemented with pcolormesh)
    - Column structure: Ground Truth + Network Prediction + Network Error (3 columns total)
    - Linear scale used for all channel Colorbars
    - Apparent resistivity (rows 1, 3) retains logarithmic y-axis for physical meaning
    """
    os.makedirs(save_dir, exist_ok=True)
    net_names = list(all_results.keys())

    if len(net_names) < 1:
        raise ValueError("all_results must contain results from at least 1 network")
    net_name = net_names[0]  # Use the first network

    ref_result = all_results[net_name]
    y_true = ref_result["y_true"][sample_idx, ...]  # Shape: (Z, Y, 4)
    epsilon = 1e-8

    # Subplot specification: 4 rows 3 columns
    fig, axes = plt.subplots(nrows=4, ncols=3, figsize=(21, 20))

    # Define Colorbar labels for each data row
    colorbar_labels = [
        r'$\log_{10}\,\rho_{xy}\,(\Omega m)$',
        r'$\phi_{xy}\,$(degree)',
        r'$\log_{10}\,\rho_{yx}\,(\Omega m)$',
        r'$\phi_{yx}\,$(degree)'
    ]

    for channel_idx in range(4):
        # -------------------------- 1. Data Preparation --------------------------
        y_true_curr = y_true[..., channel_idx]
        y_pred = all_results[net_name]["y_pred"][sample_idx, ..., channel_idx]

        # Apply log10 transformation to apparent resistivity channels (0, 2)
        if channel_idx in [0, 2]:
            y_true_curr = np.log10(y_true_curr)
            y_pred = np.log10(y_pred)

        # Combine all data to calculate Colorbar range
        all_data = np.concatenate([
            y_true_curr.flatten(),
            y_pred.flatten()
        ])
        curr_vmin = all_data.min()
        curr_vmax = all_data.max()
        curr_norm = None  # Uniform linear scale

        # -------------------------- 2. Coordinate generation --------------------------
        z_coords = all_results[net_name]["freq"]  # Frequency array
        y_coords = np.linspace(-75, 75, y_true_curr.shape[1])
        Y, Z = np.meshgrid(y_coords, z_coords)

        # -------------------------- 3. Calculate relative error --------------------------
        rel_err = np.abs(y_pred - y_true_curr) / (np.abs(y_true_curr) + epsilon) * 100
        mean_err = np.mean(rel_err)

        # -------------------------- Column 0: Ground Truth --------------------------
        ax_true = axes[channel_idx, 0]
        pc_true = ax_true.pcolormesh(
            Y, Z, y_true_curr,
            cmap="jet",
            vmin=curr_vmin,
            vmax=curr_vmax,
            shading='auto',
            edgecolors='none'
        )
        ax_true.set_yscale('log')
        ax_true.set_ylabel("Frequency (Hz)", fontsize=plt.rcParams['axes.labelsize'])
        if channel_idx == 0:
            ax_true.set_title("FDM Reference", fontsize=plt.rcParams['axes.titlesize'], fontweight="bold")

        # -------------------------- Column 1: Network Prediction --------------------------
        ax_pred = axes[channel_idx, 1]
        pc_pred = ax_pred.pcolormesh(
            Y, Z, y_pred,
            cmap="jet",
            vmin=curr_vmin,
            vmax=curr_vmax,
            shading='auto',
            edgecolors='none'
        )
        ax_pred.set_yscale('log')
        if channel_idx == 0:
            ax_pred.set_title(f"{net_name} Prediction", fontsize=plt.rcParams['axes.titlesize'], fontweight="bold")

        # -------------------------- Column 2: Network Error --------------------------
        ax_err = axes[channel_idx, 2]
        pc_err = ax_err.pcolormesh(
            Y, Z, rel_err,
            cmap="jet",
            shading='auto',
            edgecolors='none'
        )
        ax_err.set_title(f"{net_name} error: {mean_err:.2f}%", fontsize=plt.rcParams['axes.titlesize'])
        ax_err.set_yscale('log')

        # -------------------------- Colorbar Settings --------------------------
        # Data columns (columns 0-1: ground truth + prediction) share a Colorbar
        cbar_val = fig.colorbar(
            pc_true,
            ax=axes[channel_idx, :2],
            shrink=0.85,
            aspect=16
        )
        cbar_val.set_label(colorbar_labels[channel_idx], fontsize=plt.rcParams['axes.labelsize'])
        cbar_val.ax.tick_params(labelsize=plt.rcParams['xtick.labelsize'])

        # Error column (column 2) Colorbar
        cbar_err = fig.colorbar(
            pc_err,
            ax=axes[channel_idx, 2],
            shrink=0.85,
            aspect=16,
            pad=0.04
        )
        cbar_err.set_label("Relative Error (%)", fontsize=plt.rcParams['axes.labelsize'])
        cbar_err.ax.tick_params(labelsize=plt.rcParams['xtick.labelsize'])

    # Uniformly set X-axis labels for the last row (applied to all 3 columns)
    for col in range(3):
        axes[-1, col].set_xlabel("Distance (km)", fontsize=plt.rcParams['axes.labelsize'])

    # Save the figure
    save_path = os.path.join(save_dir, f"pred_error_comparison_sample{sample_idx}.png")
    plt.savefig(
        save_path,
        dpi=plt.rcParams['figure.dpi'],
        bbox_inches=plt.rcParams["savefig.bbox"],
        pad_inches=plt.rcParams["savefig.pad_inches"],
        facecolor="white"
    )
    save_path = os.path.join(save_dir, f"pred_error_comparison_sample{sample_idx}.eps")
    plt.savefig(
        save_path,
        dpi=plt.rcParams['figure.dpi'],
        bbox_inches=plt.rcParams["savefig.bbox"],
        pad_inches=plt.rcParams["savefig.pad_inches"],
        facecolor="white"
    )
    # plt.close(fig)
    print(f"Sample {sample_idx} plot saved to: {save_path}")

def plot_with_2dModel(all_results, sample_idx,save_dir="FiguresPlot"):
    os.makedirs(save_dir, exist_ok=True)
    net_names = list(all_results.keys())
    net1_name = net_names[0]  # Only need reference data from one network
    ref_result = all_results[net1_name]
    x = ref_result["x"][4, ...,0]  # Shape: (Z, Y, 4)
    x[0,:]=x[1,:]
    x[-1,:]=x[-2,:]
    x=np.log10(x)
    z_coords = np.linspace(0, 75, x.shape[1])
    y_coords = np.linspace(-75, 75, x.shape[1])
    Y, Z = np.meshgrid(y_coords, z_coords)
    fig, (axes1, axes2) = plt.subplots(nrows=1, ncols=2, figsize=(18, 8))
    pc_true = axes1.pcolormesh(
        Y, Z, x,
        cmap="jet",
        shading='auto',
        edgecolors='none'
    )
    axes1.invert_yaxis()  # Invert depth axis (surface at top)
    axes1.set_xlabel("Distance (km)", fontsize=plt.rcParams['axes.labelsize'])
    axes1.set_ylabel("Depth (km)", fontsize=plt.rcParams['axes.labelsize'])
    axes1.set_title("(a)",fontsize=26)

    x = ref_result["x"][20, ...,0]  # Shape: (Z, Y, 4)
    x[0,:]=x[1,:]
    x[-1,:]=x[-2,:]
    x=np.log10(x)
    z_coords = np.linspace(0, 75, x.shape[1])
    y_coords = np.linspace(-75, 75, x.shape[1])
    Y, Z = np.meshgrid(y_coords, z_coords)
    pc_true = axes2.pcolormesh(
        Y, Z, x,
        cmap="jet",
        shading='auto',
        edgecolors='none'
    )
    axes2.invert_yaxis()  # Invert depth axis (surface at top)
    axes2.set_xlabel("Distance (km)", fontsize=plt.rcParams['axes.labelsize'])
    axes2.set_ylabel("Depth (km)", fontsize=plt.rcParams['axes.labelsize'])
    axes2.set_title("(b)",fontsize=26)

    cbar = fig.colorbar(pc_true, ax=[axes1, axes2], orientation='horizontal', shrink=0.5, pad=0.12)
    # cbar.set_label("Conductivity (S/m)", fontsize=18)

    cbar.set_label(
        r'$\log_{10}\,\rho\,(\Omega m)$', 
        fontsize=plt.rcParams['axes.labelsize']
    )

    save_path = os.path.join(save_dir, f"pred_with_2dMode{sample_idx}.png")
    plt.savefig(
        save_path,
        dpi=plt.rcParams['figure.dpi'],
        bbox_inches=plt.rcParams["savefig.bbox"],
        pad_inches=plt.rcParams["savefig.pad_inches"],
        facecolor="white"
    )

    save_path = os.path.join(save_dir, f"pred_with_2dMode{sample_idx}.eps")
    plt.savefig(
        save_path,
        dpi=plt.rcParams['figure.dpi'],
        bbox_inches=plt.rcParams["savefig.bbox"],
        pad_inches=plt.rcParams["savefig.pad_inches"],
        facecolor="white"
    )
    plt.show(fig)

for sample_idx in range(4, 5):
    plot_with_error_comparison(all_results, sample_idx)
    # plot_with_2dModel(all_results, sample_idx)